<a href="https://colab.research.google.com/github/AbelKevinKipkosgei/pc-builder-model/blob/main/pc_builder_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# Import libraries
import pandas as pd
pd.set_option('display.max_columns', None)

In [10]:
# Load and do a first look
df = pd.read_csv('cpu.csv')

print(df.shape)
df.info()

(1413, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1413 entries, 0 to 1412
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   name               1413 non-null   object 
 1   price              547 non-null    float64
 2   core_count         1413 non-null   int64  
 3   core_clock         1413 non-null   float64
 4   boost_clock        727 non-null    float64
 5   microarchitecture  1413 non-null   object 
 6   tdp                1413 non-null   int64  
 7   graphics           621 non-null    object 
dtypes: float64(3), int64(2), object(3)
memory usage: 88.4+ KB


In [4]:
df.head(10)

,name,price,core_count,core_clock,boost_clock,microarchitecture,tdp,graphics
0,AMD Ryzen 7 9800X3D,451.50,8,4.7,5.2,Zen 5,120,Radeon
1,AMD Ryzen 7 7800X3D,340.05,8,4.2,5.0,Zen 4,120,Radeon
2,AMD Ryzen 5 7600X,170.49,6,4.7,5.3,Zen 4,105,Radeon
3,AMD Ryzen 5 9600X,204.99,6,3.9,5.4,Zen 5,65,Radeon
4,AMD Ryzen 7 7700X,242.98,8,4.5,5.4,Zen 4,105,Radeon
5,AMD Ryzen 9 9950X3D,649.99,16,4.3,5.7,Zen 5,170,Radeon
6,AMD Ryzen 5 5500,74.22,6,3.6,4.2,Zen 3,65,NaN
7,AMD Ryzen 7 9700X,305.89,8,3.8,5.5,Zen 5,65,Radeon
8,AMD Ryzen 5 5600X,158.99,6,3.7,4.6,Zen 3,65,NaN
9,AMD Ryzen 5 5600,125.61,6,3.5,4.4,Zen 3,65,NaN


In [5]:
df.isnull().sum()

,0
name,0
price,866
core_count,0
core_clock,0
boost_clock,686
microarchitecture,0
tdp,0
graphics,792


In [11]:
# Check for duplicates
print("Duplicate rows:", df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values('name').head(10)

Duplicate rows: 233


,name,price,core_count,core_clock,boost_clock,microarchitecture,tdp,graphics
1200,AMD 5150,NaN,4,1.60,NaN,Jaguar,25,Radeon HD 8400
870,AMD 5150,NaN,4,1.60,NaN,Jaguar,25,Radeon HD 8400
989,AMD 5350,NaN,4,2.05,NaN,Jaguar,25,Radeon HD 8400
441,AMD 5350,NaN,4,2.05,NaN,Jaguar,25,Radeon HD 8400
1195,AMD A10-6790K,NaN,4,4.00,4.3,Piledriver,100,Radeon HD 8670D
1204,AMD A10-6790K,NaN,4,4.00,4.3,Piledriver,100,Radeon HD 8670D
461,AMD A10-6800K,NaN,4,4.10,NaN,Piledriver,100,Radeon HD 8670D
1205,AMD A10-6800K,NaN,4,4.10,NaN,Piledriver,100,Radeon HD 8670D
1203,AMD A10-7800,NaN,4,3.50,3.9,Steamroller,65,Radeon R7 (on-die)
484,AMD A10-7800,NaN,4,3.50,3.9,Steamroller,65,Radeon R7 (on-die)


In [12]:
# Remove duplicates
print("Before:", df.shape)
df.drop_duplicates(inplace=True)
print("After:", df.shape)

Before: (1413, 8)
After: (1180, 8)


In [13]:
# Extract brand from the name column
df['brand'] = df['name'].str.split().str[0]
df['brand'].value_counts()

,count
brand,
Intel,826
AMD,354


In [14]:
# Fix boost_clock - missing means "no boost feature", not unknown
df['boost_clock'] = df['boost_clock'].fillna(df['core_clock'])
df['boost_clock'].isnull().sum()

np.int64(0)

In [15]:
# Fix graphics - missing means "no integrated graphics," not unknown
df['graphics'] = df['graphics'].fillna('None')
df['has_integrated_graphics'] = df['graphics'] != 'None'
df['graphics'].isnull().sum()

np.int64(0)

In [16]:
# Handle price - keep two versions of the table
df_priced = df[df['price'].notnull()].copy()

print("Full table (for compatibility checks):", df.shape)
print("Priced subset (for budget recommendations):", df_priced.shape)

Full table (for compatibility checks): (1180, 10)
Priced subset (for budget recommendations): (543, 10)


In [17]:
# Final check
df.isnull().sum()
df.describe()

,price,core_count,core_clock,boost_clock,tdp
count,543.000000,1180.000000,1180.000000,1180.000000,1180.000000
mean,259.217587,5.678814,3.140265,3.651197,83.959322
std,301.034259,4.747150,0.603469,0.885755,35.501831
min,12.990000,1.000000,1.100000,1.300000,20.000000
25%,94.980000,2.000000,2.800000,3.000000,65.000000
50%,169.950000,4.000000,3.200000,3.600000,71.000000
75%,311.150000,6.000000,3.600000,4.200000,100.000000
max,2699.990000,64.000000,4.700000,6.200000,280.000000


In [18]:
df.isnull().sum()

,0
name,0
price,637
core_count,0
core_clock,0
boost_clock,0
microarchitecture,0
tdp,0
graphics,0
brand,0
has_integrated_graphics,0


In [20]:
# Map microarchitecture to socket
socket_map = {
    # AMD
    'Zen 5': 'AM5',
    'Zen 4': 'AM5',
    'Zen 3': 'AM4',
    'Zen 2': 'AM4',
    'Zen+': 'AM4',
    'Zen': 'AM4',
    'Piledriver': 'FM2+',
    'Steamroller': 'FM2+',
    'Excavator': 'FM2+',
    'Bulldozer': 'FM2',
    'Jaguar': 'AM1',
    'K10': 'AM3',
    'Puma+': 'AM1',

    # Intel
    'Arrow Lake': 'LGA1851',
    'Raptor Lake Refresh': 'LGA1700',
    'Raptor Lake': 'LGA1700',
    'Alder Lake': 'LGA1700',
    'Rocket Lake': 'LGA1200',
    'Comet Lake': 'LGA1200',
    'Coffee Lake Refresh': 'LGA1151',
    'Coffee Lake': 'LGA1151',
    'Kaby Lake': 'LGA1151',
    'Skylake': 'LGA1151',
    'Cascade Lake': 'LGA3647',
    'Broadwell': 'LGA1150',
    'Haswell Refresh': 'LGA1150',
    'Haswell': 'LGA1150',
    'Ivy Bridge': 'LGA1155',
    'Sandy Bridge': 'LGA1155',
    'Westmere': 'LGA1156',
    'Nehalem': 'LGA1156',
    'Lynx': 'LGA1156',
    'Core': 'LGA775',
    'Wolfdale': 'LGA775',
    'Yorkfield': 'LGA775',
}

df['socket'] = df['microarchitecture'].map(socket_map)

# Check if anything didn't map (should be empty if our list is complete)
df[df['socket'].isnull()]['microarchitecture'].unique()

array([], dtype=object)

In [21]:
# Refresh df_priced with the updated df
df_priced = df[df['price'].notnull()].copy()
print(df_priced.shape)
df_priced.head()

(543, 11)


,name,price,core_count,core_clock,boost_clock,microarchitecture,tdp,graphics,brand,has_integrated_graphics,socket
0,AMD Ryzen 7 9800X3D,451.50,8,4.7,5.2,Zen 5,120,Radeon,AMD,True,AM5
1,AMD Ryzen 7 7800X3D,340.05,8,4.2,5.0,Zen 4,120,Radeon,AMD,True,AM5
2,AMD Ryzen 5 7600X,170.49,6,4.7,5.3,Zen 4,105,Radeon,AMD,True,AM5
3,AMD Ryzen 5 9600X,204.99,6,3.9,5.4,Zen 5,65,Radeon,AMD,True,AM5
4,AMD Ryzen 7 7700X,242.98,8,4.5,5.4,Zen 4,105,Radeon,AMD,True,AM5


In [22]:
# Save and Download
df.to_csv('cpu_clean_full.csv', index=False)
df_priced.to_csv('cpu_clean_priced.csv', index=False)

from google.colab import files
files.download('cpu_clean_full.csv')
files.download('cpu_clean_priced.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
from google.colab import files
uploaded = files.upload()

Saving motherboard.csv to motherboard.csv
